# BANA · Module 1 · Practice Pack
## Functional Programming in Python: lambda, map, filter, reduce, sorted and comprehensions

**Business case:** *Namma Brew* café, Indiranagar — same café as Session 2. Today we process a week of raw sales records.

**Why this matters for analytics:** pandas, which you will use from next week, works in exactly this style. `df["col"].apply(lambda x: ...)`, `.map()`, sorting with a `key`, and filtering rows are all functional ideas. Learn them on small lists now; they will feel natural on large tables later.

**The one big idea:** *a function is a value.* You can store it in a variable, pass it to another function, and get it back as a result.

**AI-use (AMBER):** syntax lookup is fine; the reasoning and interpretation must be your own. Expect viva questions on *why* you chose a lambda, a comprehension or a loop.

In [ ]:
# Shared data for the whole notebook — run this cell first
menu = {"Filter Coffee": 40, "Cappuccino": 120, "Masala Chai": 30, "Bun Maska": 50, "Cold Brew": 150}
bills = [460, 1250, 2080, 380, 720, 3100, 95]          # today's customer bills (₹)
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
daily_cups = [148, 132, 160, 95, 170, 210, 198]
print("Data loaded")

---
## 1 · Functions are values

Without brackets, a function name is the function itself. With brackets, you *call* it.

In [ ]:
def add_gst(amount):
    return round(amount * 1.05, 2)

print(add_gst(100))      # calling the function
print(add_gst)           # PREDICT: what does this show?

tax_rule = add_gst       # store the function in another variable
print(tax_rule(200))

In [ ]:
# A function that takes ANOTHER function as input = a "higher-order function"
def apply_to_all(rule, values):
    results = []
    for v in values:
        results.append(rule(v))
    return results

print(apply_to_all(add_gst, [100, 200, 460]))

`apply_to_all` does not care *what* the rule is. Swap in a different rule and the same code does a different job. This is the idea behind `map`, `filter`, `sorted(key=...)` and pandas `.apply()`.

---
## 2 · lambda — a one-line, nameless function

```
lambda inputs: expression_to_return
```
These two are the same:

In [ ]:
def double(x):
    return x * 2

double_lambda = lambda x: x * 2      # for demonstration only — see the style note below

print(double(21), double_lambda(21))

In [ ]:
# Lambdas can take several inputs, and can use if/else in one line
line_total   = lambda price, qty: price * qty
size_label   = lambda bill: "Large" if bill >= 1000 else "Small"

print(line_total(120, 3))
print(size_label(2080), size_label(460))

**Style rule:** if you are giving a function a name, use `def`. Lambdas are best used *inline*, passed straight into another function such as `map`, `filter` or `sorted`. A lambda can hold **one expression only** — no loops, no multiple statements.

In [ ]:
# This cell is MEANT to fail. Why?
bad = lambda x: total = x * 2

---
## 3 · map — apply a function to every item

`map(function, list)` → *"do this to each one."*

In [ ]:
with_gst = map(add_gst, bills)
print(with_gst)            # PREDICT: a list?

In [ ]:
with_gst = list(map(add_gst, bills))   # wrap in list() to see the values
print(with_gst)

in_thousands = list(map(lambda b: b / 1000, bills))
print(in_thousands)

In [ ]:
# map with TWO lists: price list and quantity list, matched position by position
prices = [120, 40, 50]
qtys   = [2, 3, 2]
print(list(map(lambda p, q: p * q, prices, qtys)))

### ⚠️ Trap: a map can only be used once

In [ ]:
converted = map(int, ["40", "120", "30"])
print(sum(converted))     # first use
print(sum(converted))     # PREDICT: same answer?

`map` (and `filter`) produce a **one-pass stream**, not a stored list. Once read, it is empty. If you need the values more than once, convert it with `list()` straight away.

---
## 4 · filter — keep only the items that pass a test

`filter(test_function, list)` keeps items where the test returns `True`.

In [ ]:
big_bills = list(filter(lambda b: b >= 1000, bills))
print(big_bills)

slow_days = list(filter(lambda pair: pair[1] < 150, zip(days, daily_cups)))
print(slow_days)

---
## 5 · reduce — combine a list into ONE value

`reduce` repeatedly combines a running result with the next item. It lives in the `functools` module, so it must be imported.

In [ ]:
from functools import reduce

total = reduce(lambda running, b: running + b, bills)
print(total, "=", sum(bills))

How it works on `[460, 1250, 2080, ...]`:
`460 + 1250 = 1710` → `1710 + 2080 = 3790` → … → final total.

In practice, prefer built-ins when they exist: `sum`, `max`, `min`, `any`, `all`. Use `reduce` only when no built-in fits.

In [ ]:
# A case where reduce helps: the highest single-day cup count AND which day it was
best = reduce(lambda a, b: a if a[1] >= b[1] else b, zip(days, daily_cups))
print(best)

---
## 6 · sorted, max and min with `key=` — the most useful lambda of all

`key` is a function that tells Python **what to compare**.

In [ ]:
# Menu items sorted by price, most expensive first
by_price = sorted(menu.items(), key=lambda item: item[1], reverse=True)
print(by_price)

# Cheapest item
print(min(menu.items(), key=lambda item: item[1]))

# Days sorted from busiest to quietest
print(sorted(zip(days, daily_cups), key=lambda d: d[1], reverse=True))

In [ ]:
# Sort by length of the item name (a different "rule", same function)
print(sorted(menu, key=len))

### ⚠️ Trap: `sorted()` vs `.sort()`

In [ ]:
prices = [120, 40, 150, 30, 50]
result = prices.sort()
print(result)       # PREDICT
print(prices)

`sorted(x)` **returns a new sorted list** and leaves the original alone. `x.sort()` **changes the original** and returns `None`. Mixing them up is a very common silent bug.

---
## 7 · Comprehensions — the Pythonic way to map and filter

A **list comprehension** does `map` and `filter` in one readable line:
```
[ expression  for item in list  if condition ]
     ↑ map                         ↑ filter
```

In [ ]:
# Same as list(map(add_gst, bills))
print([add_gst(b) for b in bills])

# Same as list(filter(lambda b: b >= 1000, bills))
print([b for b in bills if b >= 1000])

# Map AND filter together: GST-inclusive value of big bills only
print([add_gst(b) for b in bills if b >= 1000])

In [ ]:
# Dictionary comprehension: new menu with a 10% price rise, rounded to the nearest ₹5
new_menu = {item: round(price * 1.10 / 5) * 5 for item, price in menu.items()}
print(new_menu)

# Only items costing ₹50 or more
print({item: price for item, price in menu.items() if price >= 50})

**Which should I use?** Most Python programmers find comprehensions easier to read than `map`/`filter` with a lambda. Use `map` when you already have a named function (`map(int, values)`); use a comprehension when you would otherwise need a lambda.

---
## 8 · Pure functions and function factories

A **pure function** depends only on its inputs and changes nothing outside itself. Same input → same output, every time. Pure functions are easy to test and safe to reuse.

In [ ]:
# IMPURE — quietly changes a list that lives outside the function
running_log = []
def record_and_tax(amount):
    running_log.append(amount)
    return amount * 1.05

record_and_tax(100); record_and_tax(100)
print(running_log)     # PREDICT — the function changed something outside itself

# PURE — only uses its input, changes nothing else
def tax(amount):
    return amount * 1.05

### Function factories — functions that build functions

In [ ]:
def make_discount(rate):
    return lambda bill: round(bill * (1 - rate), 2)

festival_offer = make_discount(0.20)
staff_offer    = make_discount(0.50)

print(festival_offer(1000), staff_offer(1000))
print(list(map(festival_offer, bills)))

`make_discount` "remembers" the rate it was built with. This is called a **closure**. Businesses use it to create many similar rules from one template.

### ⚠️ Trap: lambdas created in a loop

In [ ]:
rates = [0.05, 0.10, 0.15]
offers = [lambda bill: bill * (1 - r) for r in rates]

print([offer(1000) for offer in offers])   # PREDICT: 950, 900, 850?

In [ ]:
# Fix: freeze the current value with a default argument
offers = [lambda bill, r=r: bill * (1 - r) for r in rates]
print([offer(1000) for offer in offers])

---
## 9 · Bridge to pandas — the same ideas on a table

In [ ]:
import pandas as pd

df = pd.DataFrame({"day": days, "cups": daily_cups})
df["revenue"]  = df["cups"].map(lambda c: c * 120)                        # map on a column
df["status"]   = df["cups"].apply(lambda c: "On target" if c >= 150 else "Below")
weekend        = df[df["day"].map(lambda d: d in ["Sat", "Sun"])]            # filter rows

display(df.sort_values("revenue", ascending=False))   # display() gives a rich table in Databricks
print(weekend)

💡 `display()` works in Databricks. If you run this notebook elsewhere (e.g., Colab), replace it with `print(df)` or just `df` on the last line.

---
# Exercises

**F1–F6 in class (pairs, swap driver/navigator) · F7–F11 homework.**
Difficulty: ⭐ warm-up · ⭐⭐ core · ⭐⭐⭐ stretch

### F1 ⭐ Predict the output — write all predictions before running anything

| # | Code | My prediction |
|---|---|---|
| a | `(lambda x: x + 10)(5)` | |
| b | `list(map(lambda x: x * 2, [1, 2, 3]))` | |
| c | `list(filter(lambda x: x % 2 == 0, range(1, 9)))` | |
| d | `sorted(["Chai", "Bun", "Cappuccino"], key=len)` | |
| e | `max([("Mon", 148), ("Sat", 210)], key=lambda d: d[1])` | |
| f | `[c for c in [148, 132, 160] if c > 140]` | |
| g | `reduce(lambda a, b: a * b, [1, 2, 3, 4])` | |
| h | `list(map(str.upper, ["tea", "bun"]))` | |
| i | `print([1, 3, 2].sort())` | |
| j | `{k: v * 2 for k, v in {"a": 1, "b": 2}.items()}` | |

In [ ]:
from functools import reduce
# Check your predictions here
print((lambda x: x + 10)(5))

### F2 ⭐ Translate between `def` and `lambda`
(a) Rewrite each `def` as a lambda passed *directly* into `map` or `filter` (no lambda variable names).
(b) Rewrite the lambda at the bottom as a properly named `def` with a docstring.

In [ ]:
cups_list = [148, 132, 160, 95, 170]

def to_revenue(cups):
    return cups * 120

def is_busy(cups):
    return cups >= 150

# (a) Your version using map(lambda ...) and filter(lambda ...):


# (b) Rewrite as a def:
loyalty_points = lambda bill: bill // 100 * 5 if bill >= 500 else 0

### F3 ⭐⭐ map — clean the raw price feed
A supplier sends prices as text with a ₹ sign and spaces. Use **one** `map` with a lambda to produce a list of integers, then use `map` again to add 5% GST.
Hint: text has `.replace()` and `.strip()` methods.

In [ ]:
raw_prices = [" ₹40", "₹120 ", " ₹30 ", "₹50", "₹150"]

# YOUR CODE

### F4 ⭐⭐ filter — find the orders that need manager approval
Each order is `(order_id, amount, payment_mode)`. Use `filter` to find orders that are **₹2,000 or more AND paid in cash**. Then write the same thing as a list comprehension. Which do you find easier to read?

In [ ]:
orders = [(101, 460, "UPI"), (102, 2500, "Cash"), (103, 2080, "Card"),
          (104, 3100, "Cash"), (105, 720, "Cash"), (106, 1999, "Cash")]

# YOUR CODE

### F5 ⭐⭐ sorted with key — rank the menu
1. Sort menu items by price, **cheapest first**.
2. Sort them **alphabetically**, ignoring capital letters (hint: `key=str.lower`).
3. Find the item with the **longest name** using `max` and a key.
4. Sort the `orders` from F4 by payment mode, and within the same mode by amount **highest first**. (Hint: a key can return a tuple, and `-amount` reverses a number.)

In [ ]:
# YOUR CODE

### F6 ⭐⭐ reduce — then decide if you needed it
1. Use `reduce` to find the total of `bills`.
2. Use `reduce` to find the **smallest** bill.
3. Use `reduce` to join the `days` list into one string `"Mon-Tue-Wed-..."`.
4. For each of the three, write the simpler built-in alternative (`sum`, `min`, `"-".join`). In one sentence: when *would* you still choose `reduce`?

In [ ]:
from functools import reduce
# YOUR CODE

### F7 ⭐⭐ Rewrite the loop — three ways
Rewrite the loop below as (a) `map` + `filter`, (b) a list comprehension. Confirm all three give the same answer.

In [ ]:
result = []
for c in daily_cups:
    if c < 150:
        result.append(150 - c)      # shortfall against target
print(result)

# (a) map + filter:

# (b) comprehension:

### F8 ⭐⭐ Fix the bugs
Each cell has **one** bug. Some crash, some are silent. Name the bug, fix it, and write a one-line comment explaining the fix.

In [ ]:
# F8a — expected: [483.0, 1312.5, ...]
gst_bills = map(lambda b: round(b * 1.05, 2), bills)
print("Bills with GST:", gst_bills)

In [ ]:
# F8b — expected: total AND count of big bills
big = filter(lambda b: b >= 1000, bills)
print("Total of big bills:", sum(big))
print("Number of big bills:", len(list(big)))

In [ ]:
# F8c — expected: the 3 highest of today's bills
today = [460, 1250, 2080, 380, 720, 3100, 95]
top3 = today.sort(reverse=True)[:3]
print(top3)

In [ ]:
# F8d — expected: 5%, 10%, 15% off a ₹2,000 bill → [1900.0, 1800.0, 1700.0]
offers = [lambda bill: bill * (1 - r) for r in [0.05, 0.10, 0.15]]
print([f(2000) for f in offers])

In [ ]:
# F8e — expected: busiest day first
busiest = sorted(zip(days, daily_cups), key=lambda d: d[0], reverse=True)
print(busiest[0])

### F9 ⭐⭐⭐ Function factory — tiered pricing rules
Write `make_tiered_discount(threshold, rate)` that **returns a function**. The returned function takes a bill and applies `rate` only if the bill is at or above `threshold`; otherwise it returns the bill unchanged.

Then build two rules and apply each to `bills` with `map`:
- `weekday_rule = make_tiered_discount(1000, 0.10)`
- `weekend_rule = make_tiered_discount(500, 0.15)`

Finally, compute how much revenue the café gives away under each rule. Which would you recommend to the owner, and what extra data would you want before deciding?

In [ ]:
def make_tiered_discount(threshold, rate):
    # YOUR CODE HERE
    pass

In [ ]:
# Test cell — do not edit
rule = make_tiered_discount(1000, 0.10)
tests = [(999, 999), (1000, 900.0), (2080, 1872.0), (95, 95)]
for bill, expected in tests:
    got = rule(bill)
    print(("✅" if got == expected else "❌"), f"bill {bill} -> {got} (expected {expected})")

### F10 ⭐⭐⭐ Mini-pipeline — from raw records to a top-3 report
Each raw record is a text line: `"day,item,quantity"`. Using **lambda, map, filter, sorted and/or comprehensions** (loops allowed only where you can justify them):

1. Parse each line into a tuple `(day, item, qty)` with `qty` as an integer
2. Add a fourth value: line revenue = `menu[item] * qty`
3. Keep only **weekend** records (Sat, Sun) and report weekend revenue
4. For the **whole week**, compute revenue per item (a dictionary) and print the **top 3 items** by revenue
5. In two sentences: what should the owner do with this information, and what is one limitation of this data?

In [ ]:
raw_sales = [
    "Mon,Cappuccino,2", "Mon,Filter Coffee,5", "Mon,Bun Maska,3",
    "Tue,Masala Chai,6", "Tue,Cappuccino,4",
    "Wed,Cold Brew,2", "Wed,Filter Coffee,7",
    "Thu,Bun Maska,1",
    "Fri,Cappuccino,6", "Fri,Cold Brew,3",
    "Sat,Cappuccino,8", "Sat,Masala Chai,10",
    "Sun,Filter Coffee,9", "Sun,Cold Brew,4", "Sun,Bun Maska,5",
]

# YOUR CODE

### F11 ⭐⭐⭐ pandas bridge
Load `raw_sales` into a DataFrame with columns `day`, `item`, `qty`. Then, using `.map()` / `.apply()` with lambdas where sensible:
1. Add a `price` column using the `menu` dictionary (hint: `Series.map` accepts a dictionary too)
2. Add a `revenue` column
3. Add a `daypart` column: `"Weekend"` for Sat/Sun, else `"Weekday"`
4. Show total revenue by `daypart` with `groupby`

In [ ]:
import pandas as pd
# YOUR CODE

---
## Debrief — exit ticket
1. In your own words: what does it mean that "a function is a value"?
2. Name one situation where you would use a **loop** instead of `map`/`filter`/a comprehension, and why.
3. Which trap today would be hardest to spot in a real report?

*Double-click to type your answers here.*

1.

2.

3.

---
## Resources to go further
| Resource | Use it for |
|---|---|
| Python *Functional Programming HOWTO* — docs.python.org/3/howto/functional.html | Official guide to map, filter, iterators, functools |
| Python tutorial §4.8 (lambda) and §5.1 (comprehensions) — docs.python.org/3/tutorial | Short official explanations |
| McKinney, *Python for Data Analysis*, 3rd ed., Ch. 3 — wesmckinney.com/book | Comprehensions, lambda and functions in a data context |
| Kaggle Learn: Python (lessons on functions and lists) — kaggle.com/learn/python | Auto-checked practice |
| Python Tutor — pythontutor.com | Watch a closure or the loop-lambda trap step by step |